# Data Science in Psychology and Neuroscience

## Class info:
* Week #6
* Day: February 24, 2026
* Time: 9:30—10:45 AM
* Location: Zoom (asynchronous recording)
* <a href="https://forms.microsoft.com/r/26vAcJWrwH">Click here to submit your attendance for Week 6, Question 11!</a>
  
## Instructor info:
* Dr. Jeremy Hogeveen
* jhogeveen@unm.edu
* Logan Hall 281 (Office Hours By Appointment)
  
## Syllabus:
* <a href="https://www.dropbox.com/scl/fi/6fs6fi4kvkwtxn7j7x8ua/PSY450650_DSPN_Spring2026_Syllabus.pdf?rlkey=148e5t4ah8q2n1daclt7mp0h6&dl=0">Download here</a>

## Today's topic:
* Wrangling a synthetic subset of the "ABCD study" dataset.

### What is the ABCD Study?

<img src="img/abcdlogo500.jpg" width=300>

The [ABCD Study](https://abcdstudy.org/) is the largest long-term study of brain development and child health in the United States. The consortium is tracking ≈12,000 youth as they grow into young adults, collecting massive amounts of data including structural and functional MRI, behavioral assessments, clinical interviews, and environmental data.

* Because of its scale, ABCD data is notoriously messy and split across dozens of text files.
* Our goals today:
    1. Wrangle demographic, clinical, and fMRI data.
    2. Pull a specific subset from the data based on a predefined list.
    3. Clean out missing data
    4. Merge the cleaned version into one mega df to rule them all!

### Where is the Data?
If you have recently updated your local git repository (e.g., by running `git pull` or downloading the latest ZIP), you should already have the synthetic dataset. Assuming you are running this notebook from inside the `code` folder, the data is located one directory up, in:
`../data/abcd_synthetic`
* Here it is online in case you need to download it manually: <a href="https://github.com/JeremyHogeveen/dspn_spring2026_github/tree/main/data/abcd_synthetic">github</a>

# Step 1: Set up our relative paths to make sure we can find the files.

In [1]:
# as in all our code!
import pandas as pd
import numpy as np
import os

# Set up our working directories relative to this notebook
script_dir = os.path.abspath('')
base_dir = os.path.dirname(script_dir)
# print(base_dir)

# getting the data directory
dir_data = os.path.join(base_dir,'data','abcd_synthetic')
# print(dir_data)
dir_outdata = os.path.join(base_dir,'data','abcd_wrangled')

# checking to see if the output dir exists, creating one if it does not
if not os.path.isdir(dir_outdata):
    os.makedirs(dir_outdata)

## 2. Exploring a Single DataFrame
* Before we load all our text files into a giant dictionary for mass wrangling, let's load just one—our demographics file (`pdem02.txt`)—to understand the structure of the data we are dealing with.
* We need to handle an annoying quirk of ABCD datasets: the first row contains column headers, but the **second row** contains string descriptions of those variables. If we don't explicitly skip it, Pandas will treat all our numeric columns as strings!

In [2]:
# first, load the demographics file
file_demo = os.path.join(dir_data,'pdem02.txt')
df_demo_sample = pd.read_csv(file_demo,sep='\t',skiprows=[1])
display(df_demo_sample.head())

# second, let's explore the raw demographics data frame
print("Shape of the raw demographics file:", df_demo_sample.shape)

# Check the data types to ensure our skiprows argument worked correctly
print(df_demo_sample.info())

,eventname,subjectkey,sex,demo_gender_id_v2,interview_age,demo_comb_income_v2,demo_fam_exp1_v2,demo_prnt_marital_v2,demo_prnt_ethn_v2,demo_prnt_race_a_v2___10,...,demo_prnt_race_a_v2___15,demo_prnt_race_a_v2___16,demo_prnt_race_a_v2___17,demo_prnt_race_a_v2___18,demo_prnt_race_a_v2___19,demo_prnt_race_a_v2___20,demo_prnt_race_a_v2___21,demo_prnt_race_a_v2___22,demo_prnt_race_a_v2___23,demo_prnt_race_a_v2___24
0,baseline_year_1_arm_1,NDAR_INV000001,1,1,2,2,1,0,0,777,...,777,2,2,777,1,2,777,1,0,777
1,baseline_year_1_arm_1,NDAR_INV000002,2,0,777,2,0,2,777,0,...,2,1,1,777,777,1,0,1,1,2
2,baseline_year_1_arm_1,NDAR_INV000003,777,0,0,1,0,2,0,2,...,777,777,2,1,1,777,0,2,2,2
3,baseline_year_1_arm_1,NDAR_INV000004,1,777,2,777,1,777,2,777,...,0,0,2,777,777,1,0,1,2,1
4,baseline_year_1_arm_1,NDAR_INV000005,777,1,0,777,1,0,2,2,...,777,1,2,777,1,2,1,1,0,0


Shape of the raw demographics file: (3400, 24)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3400 entries, 0 to 3399
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   eventname                 3400 non-null   object
 1   subjectkey                3400 non-null   object
 2   sex                       3400 non-null   int64 
 3   demo_gender_id_v2         3400 non-null   int64 
 4   interview_age             3400 non-null   int64 
 5   demo_comb_income_v2       3400 non-null   int64 
 6   demo_fam_exp1_v2          3400 non-null   int64 
 7   demo_prnt_marital_v2      3400 non-null   int64 
 8   demo_prnt_ethn_v2         3400 non-null   int64 
 9   demo_prnt_race_a_v2___10  3400 non-null   int64 
 10  demo_prnt_race_a_v2___11  3400 non-null   int64 
 11  demo_prnt_race_a_v2___12  3400 non-null   int64 
 12  demo_prnt_race_a_v2___13  3400 non-null   int64 
 13  demo_prnt_race_a_v2___14  3400 

## 3. Automating the Data Import
* We don't want to copy and paste that `pd.read_csv` command for all 8+ text files. 
    * Instead, we will use `glob` (wildcard expansion tool) to find every `*.txt` file in our directory, and a `for` loop to read them all into a single Python dictionary automatically.

In [3]:
# load glob function
from glob import glob

# load in our specific subject keys
file_subjs = os.path.join(dir_data,'subjectkeys.txt')
df_subjs = pd.read_csv(file_subjs,sep='\t')
# display(df_subjs.head())

# identify the list of input files we need
file_pattern = os.path.join(dir_data,'*.txt')
list_files = glob(file_pattern)
# print(len(list_files))
for path in list_files:
    print(os.path.basename(path))

midaparcp203.txt
abcd_lpds01.txt
abcd_mid02.txt
psychscipaper_subjectkeys.txt
subjectkeys.txt
abcd_lt01.txt
pdem02.txt
midaparc03.txt
abcd_cbcls01.txt


In [4]:
# loop through file list to wrangle more efficiently
dict_abcd_data = {}

# loop through the files
for file_path in list_files:
    if 'subjectkeys' in file_path:
        continue
    # extract just the filename without the path or '*.txt' extension
    file_name = os.path.basename(file_path).replace('.txt','')
    # print(file_name)
    # read the file and store in our dictionary
    dict_abcd_data[file_name] = pd.read_csv(file_path,sep='\t',skiprows=[1])

print("Available dataframes in our dictionary:",list(dict_abcd_data.keys()))
# display(dict_abcd_data['pdem02'])

Available dataframes in our dictionary: ['midaparcp203', 'abcd_lpds01', 'abcd_mid02', 'abcd_lt01', 'pdem02', 'midaparc03', 'abcd_cbcls01']


## 4. Extracting and Filtering the Data
* Now that all our datasets are loaded into our `dict_abcd_data` dictionary, we can pull out the specific data we need to construct our __one DF to rule them all__.
    * Specifically, we will pull the raw dataframes, filter them down to just the subjects in our `df_subjs` list using `.isin()`, and select only our variables of interest.

In [5]:
# display(df_subjs)
# display(dict_abcd_data['pdem02'])

# filter our demographics down to just the subjects we need
df_demo_raw = dict_abcd_data['pdem02']
df_demo_filt = df_demo_raw[df_demo_raw['subjectkey'].isin(df_subjs.subjectkey)]

# select the columns we need
df_demo_filt = pd.DataFrame({
    'subjectkey': df_demo_filt.subjectkey,
    'eventname': df_demo_filt.eventname,
    'sex': df_demo_filt.sex,
    'age_m': df_demo_filt.interview_age,
    'house_income': df_demo_filt.demo_comb_income_v2,
    'house_foodsecurity': df_demo_filt.demo_fam_exp1_v2})
# display(df_demo_filt.head())

# filter the site data
df_site_raw = dict_abcd_data['abcd_lt01']
df_site_filt = df_site_raw[df_site_raw['subjectkey'].isin(df_subjs.subjectkey)]

# filter the clinical CBCL data
df_cbcl_raw = dict_abcd_data['abcd_cbcls01']
df_cbcl_filt = df_cbcl_raw[df_cbcl_raw['subjectkey'].isin(df_subjs.subjectkey)]

print(f"Filtered demographics shape: {df_demo_filt.shape}")
print(f"Filtered site shape: {df_site_filt.shape}")
print(f"Filtered CBCL shape: {df_cbcl_filt.shape}")

Filtered demographics shape: (258, 6)
Filtered site shape: (258, 3)
Filtered CBCL shape: (258, 10)


## 3. The Longitudinal Challenge: Tracking Visits
* Because this is a longitudinal study, `subjectkey` is not a unique identifier on its own.
    * A single subject might have three rows of data in `df_demo_filt` representing their baseline, 6-month, and 12-month visits.

* Before we merge anything, we need to know what kind of longitudinal coverage we actually have.
    * Let's use `pd.crosstab()` to see who showed up for which visits in our demographics data.

In [6]:
# check the total number of records per visit / eventname
print(df_demo_filt['eventname'].value_counts())

# use crosstab to create a matrix of subject participation
visit_tracker = pd.crosstab(df_demo_filt['subjectkey'],df_demo_filt['eventname'])
# display(visit_tracker.tail(10))

# How many subjects completed ALL three visits?
complete_cases = visit_tracker[(visit_tracker['baseline_year_1_arm_1'] == 1) & 
                               (visit_tracker['6_month_followup'] == 1) & 
                               (visit_tracker['12_month_followup'] == 1)]

print("Subjects with complete demographic data across all 3 timepoints:",len(complete_cases))

eventname
baseline_year_1_arm_1    150
6_month_followup          65
12_month_followup         43
Name: count, dtype: int64
Subjects with complete demographic data across all 3 timepoints: 33


## 5. Merging DataFrames with real, messy data!
* When merging longitudinal datasets, we must join on **both** `subjectkey` and `eventname`.
    * If we only merged on `subjectkey`, Pandas wouldn't know which visit matched which.
    * Same principle applies to many cases where it might be handy to merge on > 1 id variable.
    * Let's revisit `inner` / `outer` / `left` / `right` join with this in mind.

In [7]:
# pull the fmri data out of our dictionary for this demo
df_mid_raw = dict_abcd_data['midaparc03']
# display(df_mid_raw.shape)

# filter to the subjects we need
df_mid_filt = df_mid_raw[df_mid_raw.subjectkey.isin(df_subjs.subjectkey)]
# display(df_mid_filt.shape)

# strategy A: the outer join (keeps all the data)
df_outer_join = pd.merge(df_demo_filt,df_mid_filt,on=['subjectkey','eventname'],how='outer')
print(df_outer_join.shape)

# strategy B: the outer join (keeps only COMPLETE cases)
df_inner_join = pd.merge(df_demo_filt,df_mid_filt,on=['subjectkey','eventname'],how='inner')
print(df_inner_join.shape)

(258, 23)
(214, 23)


## 6. Building the Mega DataFrame
* Now, let's build our final, analysis-ready dataset.
    * For this project, our primary dependent variables are in the fMRI data. Therefore, if a participant doesn't have fMRI data, we cannot use them.
    * We will use our `df_inner_join` (which already strictly filtered for complete Demo + fMRI data) as our **base** dataframe. We will then use `left` joins to attach their Site and Clinical (CBCL) data.

In [8]:
# 1. Start with our strict Demo + fMRI base
df_mega = df_inner_join.copy()

# 2. Attach Site Data
df_mega = pd.merge(df_mega, df_site_filt, on=['subjectkey', 'eventname'], how='left')

# 3. Attach Clinical (CBCL) Data
df_mega = pd.merge(df_mega, df_cbcl_filt, on=['subjectkey', 'eventname'], how='left')

# see what we have
print("Mega DF shape:", df_mega.shape)
display(df_mega.head())


Mega DF shape: (214, 32)


,subjectkey,eventname,sex,age_m,house_income,house_foodsecurity,mid_nvols_all,mid_nvols_clean,mid_mean_motion,mid_ant_rew_v_neut_l_nacc,...,mid_ant_loss_v_neut_dacc,site_id_l,cbcl_scr_syn_anxdep_t,cbcl_scr_syn_withdep_t,cbcl_scr_syn_somatic_t,cbcl_scr_syn_internal_t,cbcl_scr_syn_aggressive_t,cbcl_scr_syn_attention_t,cbcl_scr_syn_rulebreak_t,cbcl_scr_syn_external_t
0,NDAR_INV000060,baseline_year_1_arm_1,0,2,2,1,191,238,0.596849,1.369666,...,1.878495,site02,60,34,44,57,74,61,32,53
1,NDAR_INV000062,baseline_year_1_arm_1,2,2,1,0,177,220,3.100989,1.137083,...,0.064690,site01,47,61,48,72,64,35,47,49
2,NDAR_INV000074,baseline_year_1_arm_1,2,0,2,0,334,317,0.883772,0.181934,...,-1.119941,site01,46,66,32,48,42,64,75,45
3,NDAR_INV000075,baseline_year_1_arm_1,2,2,777,0,404,308,4.527369,0.553952,...,1.318909,site01,57,45,74,71,34,52,55,66
4,NDAR_INV000079,baseline_year_1_arm_1,777,1,0,1,258,242,0.437162,0.594308,...,-0.711542,site03,77,37,47,53,77,77,45,65


## 7. Final Data Cleaning & Export
Our Mega DF is assembled, but it's not quite ready for statistical modeling. We need to handle missingness within the specific survey responses.

We will drop:
1. Participants who refused to answer the family income question (coded as `777` in ABCD).
2. Participants who are missing CBCL internalizing scores (which appear as `NaN` after our left join).
3. Participants with too few clean fMRI volumes (we want high-quality scans).

In [9]:
# filter out house_income and CBCL cases
valid_income = df_mega['house_income'] != 777
valid_cbcl = df_mega['cbcl_scr_syn_internal_t'].notna()

# filtering out bad fmri data
valid_fmri = df_mega['mid_nvols_clean'] > 400

# apply booleans to our mega data frame
df_final = df_mega[valid_income & valid_cbcl & valid_fmri]
print("Subjects remaining after final exclusions:",len(df_final))

# sort by subject and event to keep our longitudinal data organized
df_final = df_final.sort_values(by=['subjectkey','eventname'])

# Note re. sort_values...: My synthetic data & various filtering requirements led this to be a bit wonky.
# Just make note of the fact that you can sort the cases in your data frame neatly
# based on one or more ID variable columns!

# save the final matrix out for statistical modeling or data visualization!
out_file = os.path.join(dir_outdata,'abcd_ses_filtered.csv')
df_final.to_csv(out_file,index=False)

# print a message to make sure you're done!
print("success! Analysis ready data saved to:",out_file)

Subjects remaining after final exclusions: 36
success! Analysis ready data saved to: /Users/jeremyhogeveen/Dropbox/Winter_2026/teaching/DSPN_Spring_2026/github/data/abcd_wrangled/abcd_ses_filtered.csv
